# Import

In [3]:
import os
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp

from glob import glob
from tqdm import tqdm
from Bio import Entrez
from Bio import SeqIO

import warnings
warnings.filterwarnings("ignore",category=FutureWarning)
warnings.filterwarnings("ignore",category=UserWarning)

# Get Anndata

In [2]:
cell_data_loc='./data/raw_data'
save_loc='./data/processed_data/gene_expression'

## frog & zebrafish

Download the raw frog & zebrafish h5ad File and map annotations from [elife paper](https://elifesciences.org/articles/66747).

The dataset is on embryogenesis.

### frog

In [ ]:
!wget -O ./data/raw_data/GSE113074_Raw_combined.annotated_counts.tsv.gz "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE113nnn/GSE113074/suppl/GSE113074_Raw_combined.annotated_counts.tsv.gz"
!gunzip ./data/raw_data/GSE113074_Raw_combined.annotated_counts.tsv.gz

In [9]:
#added progress bar for time consuming step
gc.collect()

with open(os.path.join(cell_data_loc,'frog_cell_types_mapping.txt')) as f:
    cell_types_mapping=f.readlines()
    cell_types_map={}
    for line in cell_types_mapping[1:]:
        items=line.split('\t')
        cell_types_map[items[0].strip()]=items[1].strip()
    del cell_types_mapping

filepath_frog=os.path.join(cell_data_loc,'GSE113074_Raw_combined.annotated_counts.tsv')
with open(filepath_frog) as f:
    frog_data=f.readlines()
    barcodes=[f.strip() for f in frog_data[5].split('\t')][1:]
    clusters=[f[4:].strip() if f.startswith('S') else f.strip() for f in frog_data[7].split('\t')][1:]

adata=ad.AnnData()
adata.obs['cluster']=clusters
adata.obs_names=barcodes
adata.obs_names_make_unique()
adata.obs['cluster']=adata.obs['cluster'].map(cell_types_map)
del cell_types_map

obs_selector=(~adata.obs['cluster'].isna())&(adata.obs['cluster']!='Outlier')
adata=adata[obs_selector].copy()

data=np.empty((len(frog_data[9:]),obs_selector.sum()),dtype=np.uint16)
genes=[]
Len=len(frog_data[9:][0].split('\t'))
save_index=0
for line in tqdm(frog_data[9:]):
    items=np.array(line.split('\t'))
    if len(items)!=Len:
        continue
    genes.append(items[0])
    data[save_index,:]=np.uint16(items[1:][obs_selector])
    save_index+=1
data=data[:save_index]
del obs_selector

data=sp.csr_matrix(data)
frog_data=ad.AnnData(X=data.transpose(),var=pd.DataFrame(index=genes))
del data
frog_data.obs_names=adata.obs_names
frog_data.obs['cluster']=adata.obs['cluster']
del adata

sc.pp.filter_cells(frog_data,min_genes=500)
sc.pp.filter_genes(frog_data,min_cells=10)

print(frog_data)
frog_data.write(os.path.join(save_loc,'frog.h5ad'))
del frog_data

AnnData object with n_obs × n_vars = 96935 × 24956
    obs: 'cluster', 'n_genes'
    var: 'n_cells'


### zebrafish

In [ ]:
!wget -O ./data/raw_data/WagnerScience2018.h5ad "https://kleintools.hms.harvard.edu/paper_websites/wagner_zebrafish_timecourse2018/WagnerScience2018.h5ad"

In [12]:
gc.collect()

with open(os.path.join(cell_data_loc,'zebrafish_cell_types_mapping.txt')) as f:
    cell_types_mapping=f.readlines()
cell_types_map={}
for line in cell_types_mapping[1:]:
    items=line.split('\t')
    cell_types_map[items[0].strip()]=items[1].strip()

zebrafish_data=sc.read_h5ad(os.path.join(cell_data_loc,'WagnerScience2018.h5ad'))
zebrafish_data.X=zebrafish_data.X.astype(np.uint16)

zebrafish_data.obs['cluster']=pd.Categorical([z[6:] if '-' in z else z for z in zebrafish_data.obs['ClusterName']])
zebrafish_data.obs=zebrafish_data.obs[['cluster']]

zebrafish_data.obs['cluster']=zebrafish_data.obs['cluster'].map(cell_types_map)
zebrafish_data=zebrafish_data[~zebrafish_data.obs['cluster'].isna(),:].copy()
sc.pp.filter_cells(zebrafish_data,min_genes=500)
sc.pp.filter_genes(zebrafish_data,min_cells=10)

print(zebrafish_data)
zebrafish_data.write(os.path.join(save_loc,'zebrafish.h5ad'))
del zebrafish_data

AnnData object with n_obs × n_vars = 62224 × 30028
    obs: 'cluster', 'n_genes'
    var: 'n_cells'


## human & mouse & lemur

|Dataset|Paper|
|-------|-----|
|Tabula Sapiens (human)|https://www.science.org/stoken/author-tokens/ST-495/full|
|Tabula Muris (mouse)|https://www.nature.com/articles/s41586-018-0590-4|
|Tabula Microcebus (lemur)|https://www.biorxiv.org/content/10.1101/2021.12.12.469460v1|

Download Sapiens data from https://cellxgene.cziscience.com/collections/e5f58829-1a66-40b5-a624-9046778e74f5

Download Muris data from https://figshare.com/articles/dataset/Single-cell_RNA-seq_data_from_microfluidic_emulsion_v2_/5968960/2

Download Microcebus data from https://figshare.com/articles/dataset/Tabula_Microcebus_v1_0/14468196?file=31777601

Using only blood-derived cells after process.

See 'wget's below for specific downloads

### human (~2.4GB raw)

In [ ]:
!wget -O ./data/raw_data/human_blood.h5ad "https://datasets.cellxgene.cziscience.com/988defcd-7e39-4d07-91b9-a9853af1e769.h5ad"

In [3]:
human_blood_avoid_cell_types=['naive thymus-derived CD4-positive, alpha-beta T cell']

In [4]:
gc.collect()

human_blood_data=sc.read_h5ad(os.path.join(cell_data_loc,'human_blood.h5ad'))
human_blood_data.X=human_blood_data.layers['decontXcounts'].astype(np.uint16)
human_blood_data.var_names=list(human_blood_data.var['feature_name'])
del human_blood_data.uns,human_blood_data.obsm,human_blood_data.varm,human_blood_data.layers,human_blood_data.obsp,human_blood_data.var
human_blood_data=human_blood_data[human_blood_data.obs['method']=='10X'].copy()

human_blood_data.obs['cluster']=human_blood_data.obs['cell_type']
human_blood_data.obs=human_blood_data.obs[['cluster']]
keep_mask=~human_blood_data.obs['cluster'].isin(human_blood_avoid_cell_types)
human_blood_data=human_blood_data[keep_mask,:].copy()

sc.pp.filter_cells(human_blood_data,min_genes=500)
sc.pp.filter_genes(human_blood_data,min_cells=10)
print(human_blood_data)
human_blood_data.write(os.path.join(save_loc,'human_blood.h5ad'))
del human_blood_data

AnnData object with n_obs × n_vars = 72403 × 36279
    obs: 'cluster', 'n_genes'
    var: 'n_cells'


In [ ]:
'''
all human_blood_data cell types, "##" for excluded(not using) cell types

monocyte
mature NK T cell
platelet
plasma cell
CD4-positive, alpha-beta T cell
CD8-positive, alpha-beta T cell
neutrophil
erythrocyte
hematopoietic stem cell
B cell
natural killer cell
classical monocyte
##naive thymus-derived CD4-positive, alpha-beta T cell
myeloid dendritic cell
regulatory T cell
hematopoietic precursor cell
non-classical monocyte
macrophage
intermediate monocyte
plasmacytoid dendritic cell
basophil
common myeloid progenitor
'''
pass

## mouse  (~1.5GB raw)

In [ ]:
!wget -O ./data/raw_data/mouse.zip "https://figshare.com/ndownloader/files/10700167"
!wget -O ./data/raw_data/mouse_annote.csv "https://figshare.com/ndownloader/files/10881902"

In [ ]:
!unzip ./data/raw_data/mouse.zip -d ./data/raw_data/mouse_raw

In [9]:
mouse_tissue_files=glob('./data/raw_data/mouse_raw/droplet/*')
mouse_tissue_files

['./data/raw_data/mouse_raw/droplet\\Bladder-10X_P4_3',
 './data/raw_data/mouse_raw/droplet\\Bladder-10X_P4_4',
 './data/raw_data/mouse_raw/droplet\\Bladder-10X_P7_7',
 './data/raw_data/mouse_raw/droplet\\Heart_and_Aorta-10X_P7_4',
 './data/raw_data/mouse_raw/droplet\\Kidney-10X_P4_5',
 './data/raw_data/mouse_raw/droplet\\Kidney-10X_P4_6',
 './data/raw_data/mouse_raw/droplet\\Kidney-10X_P7_5',
 './data/raw_data/mouse_raw/droplet\\Limb_Muscle-10X_P7_14',
 './data/raw_data/mouse_raw/droplet\\Limb_Muscle-10X_P7_15',
 './data/raw_data/mouse_raw/droplet\\Liver-10X_P4_2',
 './data/raw_data/mouse_raw/droplet\\Liver-10X_P7_0',
 './data/raw_data/mouse_raw/droplet\\Liver-10X_P7_1',
 './data/raw_data/mouse_raw/droplet\\Lung-10X_P7_8',
 './data/raw_data/mouse_raw/droplet\\Lung-10X_P7_9',
 './data/raw_data/mouse_raw/droplet\\Lung-10X_P8_12',
 './data/raw_data/mouse_raw/droplet\\Lung-10X_P8_13',
 './data/raw_data/mouse_raw/droplet\\Mammary_Gland-10X_P7_12',
 './data/raw_data/mouse_raw/droplet\\Mamma

In [6]:
mouse_all_ads=[]
tissue_names=[]
for tissue_file in mouse_tissue_files:
    t_ad=sc.read_10x_mtx(tissue_file)
    mouse_all_ads.append(t_ad)
    tissue_name=tissue_file.split('/')[-1]
    tissue_names.append(tissue_name)

In [10]:
mouse_all_tissues=sc.concat(mouse_all_ads,label="tissue",keys=tissue_names)
sc.pp.filter_genes(mouse_all_tissues,min_counts=500)
sc.pp.filter_cells(mouse_all_tissues,min_counts=10)

In [12]:
barcodes=pd.Series(mouse_all_tissues.obs_names).str.split('-',expand=True)[0]
tissue_ids=mouse_all_tissues.obs["tissue"].str.split('-',expand=True)[1]
new_obs_names=tissue_ids.reset_index()[1].str.cat(barcodes,sep="_")
mouse_all_tissues.obs_names=new_obs_names

In [15]:
mouse_annot=pd.read_csv('./data/raw_data/mouse_annote.csv').set_index("cell")
keep_barcodes=set(np.unique(mouse_annot.index)).intersection(set(mouse_all_tissues.obs_names))

mouse=mouse_all_tissues[list(keep_barcodes),:]
mouse.obs["cell_type"]=mouse_annot["cell_ontology_class"]
mouse.obs["cell_ontology_id"]=mouse_annot["cell_ontology_id"]
sc.pp.filter_genes(mouse,min_counts=500)
sc.pp.filter_cells(mouse,min_counts=10)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_18408\1173526173.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  mouse_annot=pd.read_csv('./data/raw_data/mouse_annote.csv').set_index("cell")


In [19]:
print(mouse.obs["cell_type"].unique())

['basal cell of epidermis' 'immature T cell' 'stromal cell' 'B cell'
 'fibroblast' 'basal cell' 'macrophage' 'mesenchymal cell'
 'alveolar macrophage' 'blood cell' 'hepatocyte' 'endothelial cell'
 'bladder urothelial cell' 'monocyte' 'skeletal muscle satellite cell'
 'T cell' 'bladder cell' 'keratinocyte' 'natural killer cell'
 'kidney collecting duct epithelial cell' 'granulocyte'
 'neuroendocrine cell' 'promonocyte' 'epithelial cell' 'late pro-B cell'
 'mesenchymal stem cell'
 'kidney loop of Henle ascending limb epithelial cell' 'proerythroblast'
 'DN1 thymic pro-T cell' nan 'basophil'
 'kidney capillary endothelial cell'
 'kidney proximal straight tubule epithelial cell' 'erythroblast'
 'hematopoietic precursor cell' 'endocardial cell' 'leukocyte'
 'granulocytopoietic cell' 'lung endothelial cell'
 'luminal epithelial cell of mammary gland' 'non-classical monocyte'
 'dendritic cell' 'erythrocyte' 'type II pneumocyte' 'kidney cell'
 'Langerhans cell' 'duct epithelial cell' 'immature

In [20]:
keep_cell_types=[
    'T cell','immature T cell','DN1 thymic pro-T cell',
    'B cell','immature B cell','early pro-B cell','late pro-B cell','Fraction A pre-pro B cell',
    'monocyte','classical monocyte','non-classical monocyte',
    'granulocyte','basophil','promonocyte','granulocytopoietic cell',
    'erythrocyte','erythroblast','proerythroblast',
    'hematopoietic precursor cell','myeloid cell',
    'natural killer cell','dendritic cell','professional antigen presenting cell'#,'blood cell','leukocyte'
]
# exclude 'blood cell' and 'leukocyte', too ambiguous

In [22]:
mouse_blood=mouse[mouse.obs["cell_type"].isin(keep_cell_types),:].copy()
mouse_blood.obs['cluster']=mouse_blood.obs['cell_type']
mouse_blood.obs=mouse_blood.obs[["cluster"]]
del mouse_blood.var["n_counts"]

sc.pp.filter_cells(mouse_blood,min_genes=500)
sc.pp.filter_genes(mouse_blood,min_cells=10)
print(mouse_blood)
mouse_blood.write(os.path.join(save_loc,'mouse_blood.h5ad'))
#del mouse_blood
#del mouse
#del mouse_all_tissues

AnnData object with n_obs × n_vars = 19082 × 12960
    obs: 'cluster', 'n_genes'
    var: 'n_cells'


### lemur (<1GB raw)

In [ ]:
!wget -O ./data/raw_data/mouse_lemur_blood.h5ad "https://figshare.com/ndownloader/files/31777601"

In [5]:
lemur_blood_avoid_cell_types=['unassigned','myeloid cell','mesothelial cell','epithelial cell','capillary endothelial cell']

In [6]:
gc.collect()

lemur_blood_data=sc.read_h5ad(os.path.join(cell_data_loc,'mouse_lemur_blood.h5ad'))
lemur_blood_data.X=lemur_blood_data.layers['raw_counts'].astype(np.uint16)
del lemur_blood_data.obsm,lemur_blood_data.uns,lemur_blood_data.layers,lemur_blood_data.var

lemur_blood_data=lemur_blood_data[lemur_blood_data.obs['method']=='10x'].copy()

lemur_blood_data.obs['cluster']=lemur_blood_data.obs['cell_ontology_class_v1']
lemur_blood_data.obs=lemur_blood_data.obs[['cluster']]
keep_mask=~lemur_blood_data.obs['cluster'].isin(lemur_blood_avoid_cell_types)
lemur_blood_data=lemur_blood_data[keep_mask,:].copy()

sc.pp.filter_cells(lemur_blood_data,min_genes=500)
sc.pp.filter_genes(lemur_blood_data,min_cells=10)

print(lemur_blood_data)
lemur_blood_data.write(os.path.join(save_loc,'lemur_blood.h5ad'))
#del lemur_blood_data

AnnData object with n_obs × n_vars = 17566 × 14847
    obs: 'cluster', 'n_genes'
    var: 'n_cells'


In [ ]:
'''
all lemur_blood_data cell types, "##" for excluded

neutrophil
CD4-positive, alpha-beta T cell
monocyte
##unassigned
CD8-positive, alpha-beta T cell
B cell
##myeloid cell
erythroid lineage cell
natural killer cell
platelet
plasma cell
##mesothelial cell
mature NK T cell
##capillary endothelial cell
##epithelial cell
hematopoietic precursor cell
megakaryocyte progenitor cell
conventional dendritic cell
erythroid progenitor cell
basophil
'''
pass

# Get Protein embeddding data

Obtaining reasonable embeddings for genes' protein sequences does not require stringent processes.

dict{gene_name:main_protein_sequence} —use any PLM model to encode the values of the dict—> dict{gene_name:embedding}, would be enough.

We specifically utilized protein embeddings archived by SATURN, see [saturn-page](https://github.com/snap-stanford/saturn).

In [ ]:
!wget -O ./data/processed_data/protein_embedding/protein_embeddings.tar.gz "http://snap.stanford.edu/saturn/data/protein_embeddings.tar.gz"
!tar -xzf./data/processed_data/protein_embedding/protein_embeddings.tar.gz
!cp ./data/processed_data/protein_embedding_export/ESM1b/frog_embedding.torch ./data/processed_data/protein_embedding/ESM1b_frog_embedding.torch
!cp ./data/processed_data/protein_embedding_export/ESM1b/zebrafish_embedding.torch ./data/processed_data/protein_embedding/ESM1b_zebrafish_embedding.torch
!cp ./data/processed_data/protein_embedding_export/ESM1b/human_embedding.torch ./data/processed_data/protein_embedding/ESM1b_human_embedding.torch
!cp ./data/processed_data/protein_embedding_export/ESM1b/mouse_embedding.torch ./data/processed_data/protein_embedding/ESM1b_mouse_embedding.torch
!cp ./data/processed_data/protein_embedding_export/ESM1b/mouse_lemur_embedding.torch ./data/processed_data/protein_embedding/ESM1b_lemur_embedding.torch